# Modelling

Этот ноутбук позволяет проводить эксперименты по подбору гиперпараметров и обучению различных моделей 

In [ ]:
import sys
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import logging

sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.logreg_model import LogRegModel
from src.models.lgbm_model import LGBMModel
from src.models.xgb_model import XGBModel
from src.models.catboost_model import CatBoostModel
from src.models.tuner import OptunaHPOTuner
from src.evaluation.metrics import compute_all_metrics

# Настройка визуализации
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

## 1. Загрузка данных

In [ ]:
DATA_PATH = Path("../data/processed/train_features.parquet")
FEATURES_JSON = Path("../data/processed/selected_features.json")

df = pd.read_parquet(DATA_PATH)

with open(FEATURES_JSON, "r") as f:
    selected_features = json.load(f)

# Оставляем только те колонки, которые реально есть в датасете
feature_cols = [c for c in selected_features if c in df.columns]
X = df[feature_cols]
y = df["TARGET"]

print(f"Dataset Shape: {X.shape}")
print(f"Target distribution: \n{y.value_counts(normalize=True)}")

## 2. Быстрый эксперимент: Baseline vs Gradient Boosting

Быстро сравним две модели "из коробки" без долгого тюнинга гиперпараметров


In [ ]:
# Logistic Regression (baseline)
lr_model = LogRegModel(params={"C": 0.1, "penalty": "l2"})
logger.info("Training Base LogReg model...")
oof_lr, models_lr = lr_model.cross_validate(X, y, n_splits=5)

# LightGBM
lgbm_model = LGBMModel(params={"n_estimators": 500, "learning_rate": 0.05})
logger.info("Training Base LGBM model...")
oof_lgbm, models_lgbm = lgbm_model.cross_validate(X, y, n_splits=5)

# Считаем финальные метрики для сравнения
metrics_lr = compute_all_metrics(y, oof_lr, prefix="LR_")
metrics_lgbm = compute_all_metrics(y, oof_lgbm, prefix="LGBM_")

comparison_df = pd.DataFrame([metrics_lr, metrics_lgbm]).T
comparison_df.columns = ['Logistic Regression', 'LightGBM']
display(comparison_df)

## 3. Выбор модели и запуск Optuna HPO
Найдем лучшие гиперпараметры для выбранной модели. Для этого меняем MODEL_TO_TUNE на любую модель из MODEL_REGISTRY

In [ ]:
import optuna.visualization as vis

MODEL_REGISTRY = {
    "logreg": LogRegModel,
    "lgbm": LGBMModel,
    "xgb": XGBModel,
    "catboost": CatBoostModel
}

MODEL_NAME = "logreg" # можно выбрать lgbm, xgb или catboost

# Автоматическое извлечение класса и параметров
ModelClass = MODEL_REGISTRY[MODEL_NAME]
model_display_name = ModelClass.__name__

tuner = OptunaHPOTuner(
    model_class=ModelClass,
    db_path="sqlite:///../optuna.db",
    study_name=f"{MODEL_NAME}_research"
)

logger.info(f"Starting HPO for {model_display_name} (key: {MODEL_NAME})")
tuner.optimize(X, y, n_trials=20)

best_params = tuner.study.best_params
print(f"Best Params for {MODEL_NAME}: {best_params}")

# Визуализация важности гиперпараметров
vis.plot_param_importances(tuner.study)

## 4. Финальная проверка (Cross-Validation) и Обучение
Чтобы графики ROC/PR были честными, нам нужны Out-of-Fold предсказания, полученные на лучших параметрах

In [ ]:
# Запуск CV для получения OOF-прогнозов
logger.info(f"Running {model_display_name} CV with best parameters...")
temp_model = ModelClass(params=best_params)
oof_preds, cv_models = temp_model.cross_validate(X, y, n_splits=5)

# Расчет метрик
final_metrics = compute_all_metrics(y, oof_preds, prefix=f"{MODEL_NAME}_")

# Обучение на полном наборе данных для интерпретации
final_model = ModelClass(params=best_params)
final_model.fit(X, y)

logger.info(f"Final model {model_display_name} is ready.")

## 5. Анализ важности признаков (Feature Importance)

In [ ]:
fi_df = final_model.get_feature_importance()

plt.figure(figsize=(10, 8))
sns.barplot(x="importance", y="feature", data=fi_df.head(20))
plt.title(f"Top 20 Features: {model_display_name} ({MODEL_NAME})")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

## 6. Оптимизация порога и Кривые (ROC, PR)

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# Данные для кривых
fpr, tpr, _ = roc_curve(y, oof_preds)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y, oof_preds)
pr_auc = final_metrics.get(f"{MODEL_NAME}_average_precision", 0)

# Бизнес-порог
thresholds = final_model.optimize_threshold(y, oof_preds, fn_cost=10.0, fp_cost=1.0)
print(f"Optimal Thresholds: {thresholds}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ROC
ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
ax1.plot([0, 1], [0, 1], color='navy', linestyle='--')
ax1.set_title(f'ROC Curve: {model_display_name}')
ax1.set_xlabel('FPR')
ax1.set_ylabel('TPR')
ax1.legend()

# PR
baseline = y.sum() / len(y)
ax2.plot(recall, precision, color='dodgerblue', lw=2, label=f'PR AUC (AP) = {pr_auc:.3f}')
ax2.axhline(y=baseline, color='red', linestyle='--', label=f'Baseline ({baseline:.3f})')
ax2.set_title(f'PR Curve: {model_display_name}')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend()

plt.show()

## 7. Калибровочная кривая
Для кредитного скоринга критически важно, чтобы предсказанная вероятность соответствовала реальной частоте дефолтов

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y, oof_preds, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label=f"{model_display_name}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual Proportion')
plt.title(f'Calibration Curve: {model_display_name}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()